<a href="https://colab.research.google.com/github/mdanmek/nida-dads-notes/blob/main/dads5001-data-tools/project/eda/04_construction_review_indicators_2569.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# การตรวจรูปแบบรายการสัญญาจ้างก่อสร้าง ปีงบประมาณ 2569

Notebook นี้ตรวจรูปแบบในรายการสัญญาวิธีเฉพาะเจาะจงที่มีวงเงินไม่เกิน 500,000 บาท

หนึ่งรายการกำหนดด้วย `รหัสโครงการ + เลขประจำตัวนิติบุคคล 13 หลัก + เลขที่สัญญา` ผลลัพธ์ใช้จัดลำดับการตรวจเอกสาร ไม่ใช่หลักฐานว่ามีการทุจริตหรือแบ่งซื้อแบ่งจ้าง


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.float_format', '{:,.2f}'.format)


In [ ]:
processed_dir = Path(
    '/content/drive/MyDrive/learning/dads/dads5001/'
    'project_1_dads5001/dataset/procurement/'
    'egp-contract/processed'
)

dataset_dir = processed_dir.parents[1]
figure_dir = processed_dir.parents[3] / 'figure'
figure_dir.mkdir(parents=True, exist_ok=True)

data_path = processed_dir / 'construction_contract_supplier_study_scope_2569.csv'
subdept_path = dataset_dir / 'egp-subdept.csv'

print(f'Contract file: {data_path}')
print(f'Subdepartment master: {subdept_path}')
print(f'Figure directory: {figure_dir}')


## 1. เปิดข้อมูลรายการสัญญา


In [ ]:
contract_data = pd.read_csv(data_path, low_memory=False)
subdept_data = pd.read_csv(subdept_path, low_memory=False)

print(f'Contract shape: {contract_data.shape}')
display(contract_data.head(5))

print(f'Subdepartment master shape: {subdept_data.shape}')
display(subdept_data.head(5))


In [ ]:
# ดาวน์โหลดฟอนต์สำหรับแสดงภาษาไทยในกราฟ
!wget -q https://github.com/Phonbopit/sarabun-webfont/raw/master/fonts/thsarabunnew-webfont.ttf
fm.fontManager.addfont('thsarabunnew-webfont.ttf')

sns.set_theme(style='whitegrid', font='TH Sarabun New')
plt.rcParams.update({
    'axes.titlesize': 18,
    'axes.titleweight': 'semibold',
    'axes.labelsize': 13,
    'xtick.labelsize': 11,
    'ytick.labelsize': 12,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.unicode_minus': False
})

BLUE = '#5B7FA3'
ORANGE = '#D9822B'
GRAY = '#B8C2CC'
TEXT = '#344054'
GRID = '#E4E7EC'


In [ ]:
project_id_column = 'รหัสโครงการ'
supplier_id_column = 'เลขประจำตัวนิติบุคคล 13 หลัก'
contract_column = 'เลขที่สัญญา'
contract_value_column = 'วงเงินงบประมาณในสัญญา (บาท)'
method_column = 'ชื่อวิธีการจัดซื้อจัดจ้าง'
agency_column = 'ชื่อหน่วยงาน'
subagency_column = 'ชื่อหน่วยงานย่อย'
province_column = 'จังหวัด'
supplier_name_column = 'ชื่อผู้ชนะการเสนอราคา'
project_name_column = 'ชื่อโครงการจัดซื้อจัดจ้าง'
scope_column = 'อยู่ในขอบเขตตรวจรูปแบบ'

agency_master_column = 'ชื่อหน่วยงานจาก master'

subdept_mapping = (
    subdept_data[[subagency_column, agency_column]]
    .dropna(subset=[subagency_column])
    .drop_duplicates(subset=[subagency_column])
    .rename(columns={agency_column: agency_master_column})
)

contract_data = contract_data.merge(
    subdept_mapping,
    on=subagency_column,
    how='left'
)

mapped_agency_count = contract_data[agency_master_column].notna().sum()

contract_data[agency_column] = (
    contract_data[agency_master_column]
    .fillna(contract_data[agency_column])
)
contract_data = contract_data.drop(columns=agency_master_column)

study_data = contract_data.loc[contract_data[scope_column]].copy()

study_summary = pd.DataFrame({
    'ข้อมูล': [
        'รายการสัญญาทั้งหมด',
        'รายการที่พบชื่อหน่วยงานใน master',
        'รายการในขอบเขตตรวจรูปแบบ'
    ],
    'จำนวนรายการสัญญา': [
        len(contract_data),
        int(mapped_agency_count),
        len(study_data)
    ]
})
study_summary['สัดส่วนจากทั้งหมด (%)'] = (
    study_summary['จำนวนรายการสัญญา'] /
    len(contract_data) * 100
)

display(study_summary)


### ตรวจชื่อหน่วยงานหลังใช้ master


In [ ]:
same_agency_mask = (
    study_data[agency_column].notna()
    & study_data[subagency_column].notna()
    & study_data[agency_column].eq(study_data[subagency_column])
)

same_agency_summary = pd.DataFrame({
    'ข้อมูล': [
        'รายการในขอบเขตตรวจรูปแบบ',
        'ชื่อหน่วยงานยังเท่ากับชื่อหน่วยงานย่อยหลังใช้ master'
    ],
    'จำนวนรายการสัญญา': [
        len(study_data),
        int(same_agency_mask.sum())
    ]
})
same_agency_summary['สัดส่วน (%)'] = (
    same_agency_summary['จำนวนรายการสัญญา'] /
    len(study_data) * 100
)

display(same_agency_summary)
display(
    study_data.loc[
        same_agency_mask,
        [agency_column, subagency_column]
    ]
    .drop_duplicates()
    .head(5)
)


`ชื่อหน่วยงาน` ใช้ค่าจาก `egp-subdept.csv` เมื่อจับคู่ `ชื่อหน่วยงานย่อย` ได้ หากจับคู่ไม่ได้จึงคงค่าจากข้อมูลสัญญาไว้ ตารางด้านบนใช้ตรวจผลหลัง mapping


## 2. Pattern 1 — รายการสัญญาใกล้เพดาน

เลือกรายการสัญญาวงเงิน 490,000–500,000 บาท แล้วเปรียบเทียบกับรายการสัญญาทั้งหมดในขอบเขตศึกษา ภายใน grain เดียวกัน 3 มุม ได้แก่ ชื่อหน่วยงาน ผู้รับจ้าง และชื่อหน่วยงานร่วมกับผู้รับจ้าง


In [ ]:
near_ceiling_data = study_data.loc[
    study_data[contract_value_column].between(
        490000,
        500000
    )
].copy()

print(
    'รายการสัญญาใกล้เพดาน: '
    f'{len(near_ceiling_data):,}'
)


In [ ]:
agency_near = (
    near_ceiling_data
    .groupby(agency_column, dropna=False)
    .agg({
        contract_column: 'size',
        contract_value_column: 'sum'
    })
    .reset_index()
    .rename(columns={
        contract_column: 'จำนวนสัญญาใกล้เพดาน',
        contract_value_column: 'มูลค่าใกล้เพดาน (บาท)'
    })
)

agency_total = (
    study_data
    .groupby(agency_column, dropna=False)
    .agg({
        contract_column: 'size',
        contract_value_column: 'sum'
    })
    .reset_index()
    .rename(columns={
        contract_column: 'จำนวนสัญญาทั้งหมด',
        contract_value_column: 'มูลค่าทั้งหมด (บาท)'
    })
)

pattern1_agency = agency_near.merge(
    agency_total,
    on=agency_column,
    how='left'
)

pattern1_agency['สัดส่วนจำนวนใกล้เพดาน (%)'] = (
    pattern1_agency['จำนวนสัญญาใกล้เพดาน'] /
    pattern1_agency['จำนวนสัญญาทั้งหมด'] * 100
)
pattern1_agency['สัดส่วนมูลค่าใกล้เพดาน (%)'] = (
    pattern1_agency['มูลค่าใกล้เพดาน (บาท)'] /
    pattern1_agency['มูลค่าทั้งหมด (บาท)'] * 100
)

pattern1_agency = pattern1_agency.sort_values(
    ['สัดส่วนจำนวนใกล้เพดาน (%)', 'จำนวนสัญญาใกล้เพดาน'],
    ascending=False
).reset_index(drop=True)

display(pattern1_agency.head(5))


In [ ]:
supplier_near = (
    near_ceiling_data
    .groupby(supplier_id_column, dropna=False)
    .agg({
        supplier_name_column: 'first',
        contract_column: 'size',
        contract_value_column: 'sum'
    })
    .reset_index()
    .rename(columns={
        contract_column: 'จำนวนสัญญาใกล้เพดาน',
        contract_value_column: 'มูลค่าใกล้เพดาน (บาท)'
    })
)

supplier_total = (
    study_data
    .groupby(supplier_id_column, dropna=False)
    .agg({
        contract_column: 'size',
        contract_value_column: 'sum'
    })
    .reset_index()
    .rename(columns={
        contract_column: 'จำนวนสัญญาทั้งหมด',
        contract_value_column: 'มูลค่าทั้งหมด (บาท)'
    })
)

pattern1_supplier = supplier_near.merge(
    supplier_total,
    on=supplier_id_column,
    how='left'
)

pattern1_supplier['สัดส่วนจำนวนใกล้เพดาน (%)'] = (
    pattern1_supplier['จำนวนสัญญาใกล้เพดาน'] /
    pattern1_supplier['จำนวนสัญญาทั้งหมด'] * 100
)
pattern1_supplier['สัดส่วนมูลค่าใกล้เพดาน (%)'] = (
    pattern1_supplier['มูลค่าใกล้เพดาน (บาท)'] /
    pattern1_supplier['มูลค่าทั้งหมด (บาท)'] * 100
)

pattern1_supplier = pattern1_supplier.sort_values(
    ['สัดส่วนจำนวนใกล้เพดาน (%)', 'จำนวนสัญญาใกล้เพดาน'],
    ascending=False
).reset_index(drop=True)

display(pattern1_supplier.head(5))


In [ ]:
agency_supplier_columns = [
    agency_column,
    supplier_id_column
]

agency_supplier_near = (
    near_ceiling_data
    .groupby(agency_supplier_columns, dropna=False)
    .agg({
        supplier_name_column: 'first',
        contract_column: 'size',
        contract_value_column: 'sum'
    })
    .reset_index()
    .rename(columns={
        contract_column: 'จำนวนสัญญาใกล้เพดาน',
        contract_value_column: 'มูลค่าใกล้เพดาน (บาท)'
    })
)

agency_supplier_total = (
    study_data
    .groupby(agency_supplier_columns, dropna=False)
    .agg({
        contract_column: 'size',
        contract_value_column: 'sum'
    })
    .reset_index()
    .rename(columns={
        contract_column: 'จำนวนสัญญาทั้งหมด',
        contract_value_column: 'มูลค่าทั้งหมด (บาท)'
    })
)

pattern1_agency_supplier = agency_supplier_near.merge(
    agency_supplier_total,
    on=agency_supplier_columns,
    how='left'
)

pattern1_agency_supplier['สัดส่วนจำนวนใกล้เพดาน (%)'] = (
    pattern1_agency_supplier['จำนวนสัญญาใกล้เพดาน'] /
    pattern1_agency_supplier['จำนวนสัญญาทั้งหมด'] * 100
)
pattern1_agency_supplier['สัดส่วนมูลค่าใกล้เพดาน (%)'] = (
    pattern1_agency_supplier['มูลค่าใกล้เพดาน (บาท)'] /
    pattern1_agency_supplier['มูลค่าทั้งหมด (บาท)'] * 100
)

pattern1_agency_supplier = pattern1_agency_supplier.sort_values(
    ['สัดส่วนจำนวนใกล้เพดาน (%)', 'จำนวนสัญญาใกล้เพดาน'],
    ascending=False
).reset_index(drop=True)

display(pattern1_agency_supplier.head(5))


In [ ]:
repeated_pairs = pattern1_agency_supplier.loc[
    pattern1_agency_supplier['จำนวนสัญญาใกล้เพดาน'].ge(3)
].copy()

pattern1_keys = set(
    repeated_pairs[[agency_column, supplier_id_column]]
    .itertuples(index=False, name=None)
)

study_data['flag_pattern_1'] = [
    (agency, supplier) in pattern1_keys
    and 490000 <= value <= 500000
    for agency, supplier, value in zip(
        study_data[agency_column],
        study_data[supplier_id_column],
        study_data[contract_value_column]
    )
]

pattern1_summary = pd.DataFrame({
    'ผลลัพธ์': [
        'รายการสัญญาใกล้เพดานทั้งหมด',
        'คู่หน่วยงาน–ผู้รับจ้างที่เกิดซ้ำอย่างน้อย 3 รายการ',
        'รายการสัญญาในคู่ที่เกิดซ้ำ'
    ],
    'จำนวน': [
        len(near_ceiling_data),
        len(repeated_pairs),
        int(study_data['flag_pattern_1'].sum())
    ]
})

display(pattern1_summary)


แต่ละตารางเปรียบเทียบรายการใกล้เพดานกับรายการสัญญาทั้งหมดภายใน grain เดียวกัน และเรียงตามสัดส่วนจำนวนใกล้เพดานจากมากไปน้อย


## 3. การพึ่งพาผู้รับจ้าง

เปรียบเทียบผู้รับจ้างหนึ่งรายกับผู้รับจ้างทั้งหมดภายในชื่อหน่วยงาน ชื่อหน่วยงานย่อย หรือจังหวัดเดียวกัน ตารางเรียงตามสัดส่วนจำนวนสัญญาจากมากไปน้อย


In [ ]:
def summarize_dependence(data, group_columns):
    supplier_summary = (
        data
        .groupby(group_columns + [supplier_id_column], dropna=False)
        .agg({
            supplier_name_column: 'first',
            contract_column: 'size',
            contract_value_column: 'sum'
        })
        .reset_index()
        .rename(columns={
            supplier_name_column: 'ผู้รับจ้าง',
            contract_column: 'จำนวนรายการของผู้รับจ้าง',
            contract_value_column: 'มูลค่าของผู้รับจ้าง (บาท)'
        })
    )

    group_summary = (
        data
        .groupby(group_columns, dropna=False)
        .agg({
            contract_column: 'size',
            contract_value_column: 'sum'
        })
        .reset_index()
        .rename(columns={
            contract_column: 'จำนวนรายการทั้งหมด',
            contract_value_column: 'มูลค่ารวมทั้งหมด (บาท)'
        })
    )

    summary = supplier_summary.merge(
        group_summary,
        on=group_columns,
        how='left'
    )

    summary['สัดส่วนจำนวนรายการ (%)'] = (
        summary['จำนวนรายการของผู้รับจ้าง'] /
        summary['จำนวนรายการทั้งหมด'] * 100
    )
    summary['สัดส่วนมูลค่า (%)'] = (
        summary['มูลค่าของผู้รับจ้าง (บาท)'] /
        summary['มูลค่ารวมทั้งหมด (บาท)'] * 100
    )

    return summary


### Pattern 2 — ชื่อหน่วยงาน + ผู้รับจ้าง

เลือกผู้รับจ้างที่มีอย่างน้อย 10 รายการ และมีสัดส่วนทั้งจำนวนสัญญาและมูลค่าอย่างน้อย 75% ภายในชื่อหน่วยงานเดียวกัน


In [ ]:
agency_data = study_data.loc[
    study_data[agency_column].notna()
    & study_data[supplier_id_column].notna()
].copy()

agency_dependence = summarize_dependence(
    agency_data,
    [agency_column]
)

pattern2_pairs = (
    agency_dependence.loc[
        agency_dependence['จำนวนรายการของผู้รับจ้าง'].ge(10)
        & agency_dependence['สัดส่วนจำนวนรายการ (%)'].ge(75)
        & agency_dependence['สัดส่วนมูลค่า (%)'].ge(75)
    ]
    .sort_values(
        ['สัดส่วนจำนวนรายการ (%)', 'จำนวนรายการของผู้รับจ้าง'],
        ascending=False
    )
    .reset_index(drop=True)
)

pattern2_keys = set(
    pattern2_pairs[[agency_column, supplier_id_column]]
    .itertuples(index=False, name=None)
)

study_data['flag_pattern_2'] = [
    (agency, supplier) in pattern2_keys
    for agency, supplier in zip(
        study_data[agency_column],
        study_data[supplier_id_column]
    )
]

display(pattern2_pairs.head(5))


แต่ละแถวแสดงจำนวนสัญญาและมูลค่าของผู้รับจ้าง พร้อม `% total` เมื่อเทียบกับจำนวนสัญญาและมูลค่ารวมของชื่อหน่วยงานนั้น ตารางเรียงตามสัดส่วนจำนวนสัญญาจากมากไปน้อย


### Pattern 3 — ชื่อหน่วยงานย่อย + ผู้รับจ้าง

ใช้เฉพาะ `ชื่อหน่วยงานย่อย + ผู้รับจ้าง` เป็น grain เลือกผู้รับจ้างที่มีอย่างน้อย 10 รายการ และมีสัดส่วนทั้งจำนวนสัญญาและมูลค่าอย่างน้อย 75% ภายในชื่อหน่วยงานย่อยเดียวกัน


In [ ]:
subagency_data = study_data.loc[
    study_data[subagency_column].notna()
    & study_data[supplier_id_column].notna()
].copy()

subagency_dependence = summarize_dependence(
    subagency_data,
    [subagency_column]
)

pattern3_pairs = (
    subagency_dependence.loc[
        subagency_dependence['จำนวนรายการของผู้รับจ้าง'].ge(10)
        & subagency_dependence['สัดส่วนจำนวนรายการ (%)'].ge(75)
        & subagency_dependence['สัดส่วนมูลค่า (%)'].ge(75)
    ]
    .sort_values(
        ['สัดส่วนจำนวนรายการ (%)', 'จำนวนรายการของผู้รับจ้าง'],
        ascending=False
    )
    .reset_index(drop=True)
)

pattern3_keys = set(
    pattern3_pairs[[subagency_column, supplier_id_column]]
    .itertuples(index=False, name=None)
)

study_data['flag_pattern_3'] = [
    (subagency, supplier) in pattern3_keys
    for subagency, supplier in zip(
        study_data[subagency_column],
        study_data[supplier_id_column]
    )
]

display(pattern3_pairs.head(5))


แต่ละแถวเทียบผู้รับจ้างหนึ่งรายกับผู้รับจ้างทั้งหมดภายในชื่อหน่วยงานย่อยเดียวกัน โดยไม่ใช้ชื่อหน่วยงานร่วมใน grain


### Pattern 4 — จังหวัด + ผู้รับจ้าง

พิจารณาจังหวัดที่มีอย่างน้อย 20 รายการ ผู้รับจ้างมีอย่างน้อย 5 รายการ และมีสัดส่วนทั้งจำนวนสัญญาและมูลค่าอย่างน้อย 50% ภายในจังหวัดเดียวกัน


In [ ]:
province_data = study_data.loc[
    study_data[province_column].notna()
    & study_data[supplier_id_column].notna()
].copy()

province_dependence = summarize_dependence(
    province_data,
    [province_column]
)

eligible_province_pairs = province_dependence.loc[
    province_dependence['จำนวนรายการทั้งหมด'].ge(20)
    & province_dependence['จำนวนรายการของผู้รับจ้าง'].ge(5)
].copy()

pattern4_pairs = (
    eligible_province_pairs.loc[
        eligible_province_pairs['สัดส่วนจำนวนรายการ (%)'].ge(50)
        & eligible_province_pairs['สัดส่วนมูลค่า (%)'].ge(50)
    ]
    .sort_values(
        ['สัดส่วนจำนวนรายการ (%)', 'จำนวนรายการของผู้รับจ้าง'],
        ascending=False
    )
    .reset_index(drop=True)
)

pattern4_keys = set(
    pattern4_pairs[[province_column, supplier_id_column]]
    .itertuples(index=False, name=None)
)

study_data['flag_pattern_4'] = [
    (province, supplier) in pattern4_keys
    for province, supplier in zip(
        study_data[province_column],
        study_data[supplier_id_column]
    )
]

pattern4_summary = pd.DataFrame({
    'ผลลัพธ์': ['คู่จังหวัด–ผู้รับจ้างที่ผ่านเกณฑ์'],
    'จำนวน': [len(pattern4_pairs)]
})

display(pattern4_summary)
display(pattern4_pairs)


แต่ละแถวแสดงจำนวนสัญญาและมูลค่าของผู้รับจ้าง พร้อม `% total` เมื่อเทียบกับจังหวัดนั้น หากตารางว่าง หมายถึงไม่มีคู่จังหวัด–ผู้รับจ้างที่ผ่านเกณฑ์ที่กำหนด


## 4. รายการสัญญาตรวจสอบลำดับแรก

ซ้อนทับ Pattern 1 กับ Pattern 2 หรือ Pattern 3 ส่วน Pattern 4 ใช้เป็นข้อมูลประกอบระดับจังหวัด


In [ ]:
study_data['priority_review'] = (
    study_data['flag_pattern_1']
    & (
        study_data['flag_pattern_2']
        | study_data['flag_pattern_3']
    )
)

result_summary = pd.DataFrame({
    'เงื่อนไข': [
        'Pattern 1: เกิดซ้ำใกล้เพดาน',
        'Pattern 2: พึ่งพาในชื่อหน่วยงาน',
        'Pattern 3: พึ่งพาในชื่อหน่วยงานย่อย',
        'Pattern 4: พึ่งพาในจังหวัด',
        'เข้า Pattern 1 และ Pattern 2 หรือ 3'
    ],
    'จำนวนรายการสัญญา': [
        int(study_data['flag_pattern_1'].sum()),
        int(study_data['flag_pattern_2'].sum()),
        int(study_data['flag_pattern_3'].sum()),
        int(study_data['flag_pattern_4'].sum()),
        int(study_data['priority_review'].sum())
    ]
})

display(result_summary)


In [ ]:
plot_data = result_summary.sort_values('จำนวนรายการสัญญา')
colors = [
    ORANGE if condition == 'เข้า Pattern 1 และ Pattern 2 หรือ 3' else BLUE
    for condition in plot_data['เงื่อนไข']
]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(
    plot_data['เงื่อนไข'],
    plot_data['จำนวนรายการสัญญา'],
    color=colors
)
ax.bar_label(
    bars,
    labels=[f'{value:,.0f}' for value in plot_data['จำนวนรายการสัญญา']],
    padding=5,
    color=TEXT
)
ax.set_title('จำนวนรายการสัญญาที่เข้าแต่ละ Pattern')
ax.set_xlabel('จำนวนรายการสัญญา')
ax.set_ylabel('')
ax.grid(axis='x', color=GRID)
ax.grid(axis='y', visible=False)
sns.despine(ax=ax, left=True, bottom=True)
fig.tight_layout()
plt.show()


In [ ]:
png_path = figure_dir / 'fig04_01_contract_records_by_pattern.png'
svg_path = figure_dir / 'fig04_01_contract_records_by_pattern.svg'

fig.savefig(png_path, dpi=180, bbox_inches='tight', facecolor='white')
fig.savefig(svg_path, bbox_inches='tight', facecolor='white')

print(f'Saved: {png_path}')
print(f'Saved: {svg_path}')


In [ ]:
priority_contracts = (
    study_data.loc[
        study_data['priority_review'],
        [
            project_id_column,
            contract_column,
            project_name_column,
            agency_column,
            subagency_column,
            province_column,
            supplier_id_column,
            supplier_name_column,
            contract_value_column,
            'flag_pattern_1',
            'flag_pattern_2',
            'flag_pattern_3',
            'flag_pattern_4'
        ]
    ]
    .sort_values(
        [agency_column, supplier_name_column, contract_value_column],
        ascending=[True, True, False]
    )
    .reset_index(drop=True)
)

print(f'รายการสัญญาตรวจสอบลำดับแรก: {len(priority_contracts):,}')
display(priority_contracts.head(5))


### ตรวจอะไรต่อจากรายการลำดับแรก

ใช้ตารางรายสัญญาเพื่อเปิดเอกสารต้นทางและตรวจร่วมกันอย่างน้อย 4 เรื่อง:

1. ขอบเขตงานและสถานที่ก่อสร้างของสัญญาที่เกิดซ้ำ
2. วิธีจัดทำราคากลางและรายการประมาณราคา
3. ผู้เสนอราคาและเอกสารเสนอราคาของแต่ละสัญญา
4. ความเชื่อมโยงของงานที่อาจเป็นงานเดียวกันหรือแบ่งออกเป็นหลายสัญญา

Pattern เป็นเพียงตัวชี้ตำแหน่งที่ควรเริ่มตรวจ ข้อสรุปต้องอาศัยเอกสารของแต่ละสัญญา


## 5. บันทึกผลลัพธ์


In [ ]:
contract_flags = contract_data.copy()

flag_columns = [
    project_id_column,
    supplier_id_column,
    contract_column,
    'flag_pattern_1',
    'flag_pattern_2',
    'flag_pattern_3',
    'flag_pattern_4',
    'priority_review'
]

contract_flags = contract_flags.merge(
    study_data[flag_columns],
    on=[project_id_column, supplier_id_column, contract_column],
    how='left'
)

for column in [
    'flag_pattern_1',
    'flag_pattern_2',
    'flag_pattern_3',
    'flag_pattern_4',
    'priority_review'
]:
    contract_flags[column] = (
        contract_flags[column]
        .astype('boolean')
        .fillna(False)
    )

output_objects = {
    'construction_contract_review_indicators_2569.csv': contract_flags,
    'priority_review_contracts_2569.csv': priority_contracts,
    'near_500k_by_agency_2569.csv': pattern1_agency,
    'near_500k_by_supplier_2569.csv': pattern1_supplier,
    'near_500k_by_agency_supplier_2569.csv': pattern1_agency_supplier,
    'repeated_near_500k_agency_supplier_2569.csv': repeated_pairs,
    'agency_supplier_dependence_2569.csv': pattern2_pairs,
    'subagency_supplier_dependence_2569.csv': pattern3_pairs,
    'province_supplier_dependence_2569.csv': pattern4_pairs
}

export_records = []

for file_name, output_data in output_objects.items():
    output_path = processed_dir / file_name
    output_data.to_csv(output_path, index=False, encoding='utf-8-sig')
    export_records.append({
        'ไฟล์': file_name,
        'จำนวนแถว': len(output_data)
    })

export_summary = pd.DataFrame(export_records)
display(export_summary)
